# 纯现货公司包：零段向下（0→-1）

本 Notebook 固定只运行 `down` 侧。公司端在本侧完整大候选池内用自己的 Development+Validation 独立选择 Top1，随后才展示冻结后的 Test。运行时只读取 `COMPANY_SPOT_PATH` 指向的一个原始现货文件，并在包内计算八状态；另一侧及任何状态旁路文件都不参与运行。

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import Markdown, display

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'pool_registry.py').is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from spot_panel import load_spot_panel
from zero_transfer.spot_eight_state import EIGHT_STATE_NAMES
from runtime_paths import resolve_output_dir, resolve_spot_path
from company_runner import run_company_side

SIDE = 'down'
SPOT_PATH = resolve_spot_path()
OUTPUT_DIR = resolve_output_dir(PACKAGE_ROOT) / '01_down'
print('阶段 0/5：读取唯一现货输入并构造内部八状态', flush=True)
spot, panel, spot_audit = load_spot_panel(SPOT_PATH)
display(pd.DataFrame([
    {'项': '侧别', '结果': SIDE},
    {'项': '唯一数据输入', '结果': str(SPOT_PATH)},
    {'项': '内部八状态引擎', '结果': spot_audit['state_engine']},
    {'项': '八状态名称', '结果': ', '.join(EIGHT_STATE_NAMES)},
    {'项': '输入行数', '结果': spot_audit['input_rows']},
    {'项': '实际状态值计数', '结果': spot_audit['state_counts']},
    {'项': '非现货输入', '结果': spot_audit['non_spot_inputs_read']},
]))
display(Markdown('## 输入字段与内部状态审计'))
display(pd.DataFrame({'raw_columns': [spot_audit['raw_columns_seen']], 'used_columns': [spot_audit['spot_columns_used']], 'future_values_used_in_state': [spot_audit['future_values_used_in_state']]}).T)

## 冻结前扫描进度与候选概览

下一格会打印 5 个阶段的实时进度；扫描结束后，冻结前摘要和 Top20 会先展示，Test 仍只在冻结之后计算。

In [ ]:
print('开始本侧独立候选扫描；Test 在冻结前保持锁定。', flush=True)
freeze = run_company_side(SIDE, spot, panel, OUTPUT_DIR, show_progress=True)
assert freeze['test_used_for_selection'] is False
print('本侧候选扫描和冻结完成；下一格显示冻结后完整结果。', flush=True)

In [ ]:
display(Markdown('## 冻结前：Development+Validation 候选概览'))
pre = json.loads((OUTPUT_DIR / 'company_pre_freeze.json').read_text(encoding='utf-8'))
display(pd.DataFrame({
    '项': ['本侧输入候选数', 'Dev/Val 可计算数', '转移主排序基线 Top1', '最终冻结候选', '选择策略', '全网格质量审计候选数', 'Dev/Val 事件质量通过数', 'Dev/Val 事件+持仓双质量通过数', 'Test 是否仍锁定'],
    '结果': [pre['candidate_count_input'], pre['candidate_count_computable'], pre['baseline_top_candidate_id'], pre['selected_candidate_preview']['candidate_id'], pre['selection_policy'], pre['state_quality_shortlist_count'], pre['state_quality_gate_candidate_count'], pre['holding_quality_gate_candidate_count'], pre['test_locked_until_after_freeze']],
}))
display(pd.read_csv(OUTPUT_DIR / 'company_top20.csv').head(20))
display(pd.DataFrame([pre['development'], pre['validation']], index=['development', 'validation']))

## 最终冻结输出

最后一个代码格固定输出官方全体 Top1 的冻结参数和全周期/分周期指标，随后展示每个核心逻辑的局部 Top1 及 Test 诊断；局部表只作观察，适合直接截图。

In [ ]:
display(Markdown('## 最后一格：最终冻结参数与全周期/分周期指标'))
param = pd.DataFrame({
    '参数': ['side', 'candidate_id', 'baseline_top_candidate_id', 'selection_policy', 'state_quality_override_applied', 'state_quality_shortlist_count', 'state_quality_dominator_count', 'source_version', 'core_logic_name', 'method_key', 'score_variant_number', 'min_zero_age', 'entry_quantile', 'confirmation_days', 'holding_package', 'threshold_from_development', 'test_used_for_selection'],
    '值': [freeze['side'], freeze['candidate_id'], freeze['baseline_top_candidate_id'], freeze['selection_policy'], freeze['state_quality_override_applied'], freeze['state_quality_shortlist_count'], freeze['state_quality_dominator_count'], freeze['source_version'], freeze['core_logic_name'], freeze['method_key'], freeze['score_variant_number'], freeze['min_zero_age'], freeze['entry_quantile'], freeze['confirmation_days'], freeze['holding_package'], freeze['threshold_from_development'], freeze['test_used_for_selection']],
})
display(param)
phase_rows = []
for name in ['full', 'development', 'validation', 'test']:
    row = {'period': name, **freeze[name], 'start': freeze['phase_dates'][name]['start'], 'end': freeze['phase_dates'][name]['end']}
    phase_rows.append(row)
metrics = pd.DataFrame(phase_rows)
metric_columns = ['start', 'end', 'selected_days', 'eligible_days', 'coverage', 'mean_directional_o2o_h1', 'mean_directional_o2o_h3', 'h1_hit_rate', 'rank_ic', 'path_consistency_h1_h3', 'improvement_vs_all_eligible_zero', 'target_transition_rate', 'opposite_transition_rate', 'transition_purity', 'target_transition_lift_vs_eligible', 'rapid_restore_rate', 'test_used_for_selection']
metrics_display = metrics.set_index('period')[metric_columns].T
display(metrics_display)
display(Markdown('## 最终冻结候选自己的状态质量（只作审查，不读取 Test 来选参）'))
state_rows = []
for phase_name, state_metrics in freeze.get('state_quality', {}).items():
    state_rows.append({'period': phase_name, **state_metrics})
display(pd.DataFrame(state_rows).set_index('period').T)
display(Markdown('## 最终冻结候选自己的实际持仓段质量（不把 entry 事件段混入）'))
holding_rows = []
for phase_name, holding_metrics in freeze.get('holding_quality', {}).items():
    holding_rows.append({'period': phase_name, **holding_metrics})
display(pd.DataFrame(holding_rows).set_index('period').T)
display(Markdown('## 上传包内每个逻辑的局部 Top1（诊断）'))
logic = pd.read_csv(OUTPUT_DIR / 'company_logic_top1.csv')
for phase in ['development', 'validation', 'full', 'test']:
    logic[f'{phase}_h1_bp'] = logic[f'{phase}_mean_directional_o2o_h1'] * 10000.0
    logic[f'{phase}_h3_bp'] = logic[f'{phase}_mean_directional_o2o_h3'] * 10000.0
logic_display = logic[['method_key', 'core_logic_name', 'source_version', 'candidate_id', 'development_h1_bp', 'validation_h1_bp', 'full_h1_bp', 'test_h1_bp', 'test_h3_bp', 'test_selected_days', 'test_h1_hit_rate', 'test_target_transition_rate', 'test_opposite_transition_rate', 'test_transition_purity', 'test_rapid_restore_rate', 'test_used_for_selection']].copy()
logic_display.columns = ['logic', '核心逻辑', '来源版本', '逻辑局部 Top1', 'Dev H1(bp)', 'Val H1(bp)', 'Full H1(bp)', 'Test H1(bp)', 'Test H3(bp)', 'Test天数', 'Test命中率', 'Test目标转移率', 'Test反侧转移率', 'Test转移纯度', 'Test快速回零率', 'Test参与选择']
display(logic_display.round({'Dev H1(bp)': 2, 'Val H1(bp)': 2, 'Full H1(bp)': 2, 'Test H1(bp)': 2, 'Test H3(bp)': 2, 'Test命中率': 3, 'Test目标转移率': 3, 'Test反侧转移率': 3, 'Test转移纯度': 3, 'Test快速回零率': 3}))
display(Markdown('上表每个逻辑只用 Development+Validation 选局部 Top1；Test 仅在局部冻结后计算，不参与全体候选池或状态质量护栏。'))
display(Markdown('## 跨逻辑信号重合审计（诊断）'))
overlap_path = OUTPUT_DIR / 'company_logic_signal_overlap.csv'
if overlap_path.is_file() and overlap_path.stat().st_size > 1:
    overlap = pd.read_csv(overlap_path)
    overlap_display = overlap.sort_values(['jaccard_overlap', 'overlap_rate_of_smaller'], ascending=False).head(20).copy()
    overlap_display = overlap_display[['left_method_key', 'left_source_version', 'right_method_key', 'right_source_version', 'left_selected_days', 'right_selected_days', 'intersection_days', 'jaccard_overlap', 'overlap_rate_of_smaller', 'binary_phi_correlation', 'exact_same_path']]
    overlap_display.columns = ['左逻辑', '左来源', '右逻辑', '右来源', '左信号天数', '右信号天数', '重合天数', 'Jaccard重合', '较小侧重合率', '二元Phi相关', '完全同路径']
else:
    overlap_display = pd.DataFrame(columns=['左逻辑', '左来源', '右逻辑', '右来源', '左信号天数', '右信号天数', '重合天数', 'Jaccard重合', '较小侧重合率', '二元Phi相关', '完全同路径'])
display(overlap_display.round({'Jaccard重合': 3, '较小侧重合率': 3, '二元Phi相关': 3}))
display(Markdown('该表只审计冻结后各逻辑局部 Top1 的实际入场日路径，不参与任何选择；完整配对结果见 `company_logic_signal_overlap.csv/json`。Jaccard=0.50 只是提示阈值，不是淘汰规则。文件：`company_logic_top1.csv/json`、`company_freeze.json`、`company_top20.csv`。'))
adjustment_path = OUTPUT_DIR.parent / 'company_state_adjustment_diagnostics.json'
if not adjustment_path.is_file():
    adjustment_path = OUTPUT_DIR / 'company_state_adjustment_diagnostics.json'
if adjustment_path.is_file():
    adjustment = json.loads(adjustment_path.read_text(encoding='utf-8'))
    display(Markdown('## 冻结后状态重构审计（不参与筛选）'))
    display(Markdown('正式三状态只在下侧/上侧实际 `entry_signal` 当天把基准 0 改为 -1/+1；同一 0 段后续日期不会因一次信号被整体延续重标。另行展示实际 `holding_signal` 的连续持仓段，以及把信号延伸到 0 段末尾的 persistent 仅审计对照。该审计只检查段长度、一天持仓占比、胜率和三状态收益分布，不改变冻结参数。'))
    display(pd.DataFrame([{
        'pair_complete': adjustment.get('pair_complete'),
        'down_entry_signal_days': adjustment.get('signal_rows', {}).get('down_signal_days'),
        'up_entry_signal_days': adjustment.get('signal_rows', {}).get('up_signal_days'),
        'down_holding_days': adjustment.get('holding_rows', {}).get('down_holding_days'),
        'up_holding_days': adjustment.get('holding_rows', {}).get('up_holding_days'),
        'conflict_days': adjustment.get('signal_rows', {}).get('conflict_days'),
        'down_signal_outside_zero_days': adjustment.get('signal_rows', {}).get('down_signal_outside_zero_days'),
        'up_signal_outside_zero_days': adjustment.get('signal_rows', {}).get('up_signal_outside_zero_days'),
    }]))
    display(Markdown('### 状态重构自动判定（仅审查，不参与筛选）'))
    display(pd.DataFrame([adjustment.get('quality_assessment', {})]))
    period_name = 'full'
    period = adjustment.get('periods', {}).get(period_name, {})
    segment_rows = []
    segment_return_rows = []
    return_rows = []
    modes = ['base', 'event_relabel', 'persistent_relabel', 'persistent_relabel_conflict_continue', 'persistent_relabel_gap_bridge_1', 'persistent_relabel_gap_bridge_2']
    mode_labels = {'base': 'base', 'event_relabel': 'event_relabel', 'persistent_relabel': 'persistent_relabel', 'persistent_relabel_conflict_continue': 'conflict_continue_causal_audit', 'persistent_relabel_gap_bridge_1': 'gap_bridge_1_retrospective', 'persistent_relabel_gap_bridge_2': 'gap_bridge_2_retrospective'}
    for mode in modes:
        for state in ['-1', '0', '1']:
            seg = period.get(mode, {}).get('segment_stats', {}).get(state, {})
            segret = period.get(mode, {}).get('segment_return_stats', {}).get(state, {})
            ret = period.get(mode, {}).get('return_stats', {}).get(state, {})
            label = mode_labels.get(mode, mode)
            segment_rows.append({'mode': label, 'state': state, 'segments': seg.get('segment_count'), 'mean_length': seg.get('mean_length'), 'median_length': seg.get('median_length'), 'one_day_share': seg.get('one_day_share'), 'max_length': seg.get('max_length')})
            segment_return_rows.append({'mode': label, 'state': state, 'segments': segret.get('n'), 'mean_segment_return_bp': (segret.get('mean_segment_directional_return') or float('nan')) * 10000.0, 'median_segment_return_bp': (segret.get('median_segment_directional_return') or float('nan')) * 10000.0, 'segment_win_rate': segret.get('segment_win_rate_directional'), 'p05_segment_bp': (segret.get('p05_segment_directional_return') or float('nan')) * 10000.0, 'p95_segment_bp': (segret.get('p95_segment_directional_return') or float('nan')) * 10000.0})
            return_rows.append({'mode': label, 'state': state, 'n': ret.get('n'), 'mean_return_bp': (ret.get('mean_return') or float('nan')) * 10000.0, 'mean_directional_bp': (ret.get('mean_directional_return') or float('nan')) * 10000.0, 'win_rate': ret.get('win_rate_directional'), 'p05_bp': (ret.get('p05_return') or float('nan')) * 10000.0, 'p95_bp': (ret.get('p95_return') or float('nan')) * 10000.0})
    display(Markdown('### 全周期持仓段长度'))
    display(pd.DataFrame(segment_rows).round({'mean_length': 2, 'median_length': 2, 'one_day_share': 3}))
    display(Markdown('### 全周期持仓段胜率与段收益分布（bp）'))
    display(pd.DataFrame(segment_return_rows).round({'mean_segment_return_bp': 2, 'median_segment_return_bp': 2, 'segment_win_rate': 3, 'p05_segment_bp': 2, 'p95_segment_bp': 2}))
    display(Markdown('### 全周期三个状态持仓收益分布（bp）'))
    display(pd.DataFrame(return_rows).round({'mean_return_bp': 2, 'mean_directional_bp': 2, 'win_rate': 3, 'p05_bp': 2, 'p95_bp': 2}))
    display(Markdown('### 实际冻结持仓路径的段胜率（不把状态持续重标混入）'))
    def _bp(value):
        return value * 10000.0 if value is not None else float('nan')
    holding_rows = []
    for side_name, side_label in [('down', '下侧 -1'), ('up', '上侧 +1')]:
        for phase_name in ['full', 'development', 'validation', 'test']:
            hp = adjustment.get('holding_paths', {}).get(side_name, {}).get(phase_name, {})
            hs = hp.get('segment_stats', {})
            hsr = hp.get('segment_return_stats', {})
            hr = hp.get('return_stats', {})
            holding_rows.append({
                'side': side_label,
                'period': phase_name,
                'holding_days': hp.get('holding_days'),
                'coverage': hp.get('holding_coverage'),
                'segments': hs.get('segment_count'),
                'mean_length': hs.get('mean_length'),
                'one_day_share': hs.get('one_day_share'),
                'segment_win_rate': hsr.get('segment_win_rate_directional'),
                'mean_segment_return_bp': _bp(hsr.get('mean_segment_directional_return')),
                'daily_win_rate': hr.get('win_rate_directional'),
                'daily_mean_return_bp': _bp(hr.get('mean_directional_return')),
            })
    display(pd.DataFrame(holding_rows).round({'coverage': 3, 'mean_length': 2, 'one_day_share': 3, 'segment_win_rate': 3, 'mean_segment_return_bp': 2, 'daily_win_rate': 3, 'daily_mean_return_bp': 2}))
    display(Markdown('正式状态口径是 `event_relabel`：只改 entry_signal 当天；段胜率与持仓时间口径则只按实际冻结 `holding_signal` 的连续段计算。`persistent_relabel` 是把 entry 信号延伸到整个基准 0 段的非生产审计，两者不能混用。'))
    display(Markdown('### 持续持仓重标相对基线变化'))
    for improvement_name in ['persistent_vs_base', 'gap_bridge_1_vs_persistent', 'gap_bridge_2_vs_persistent']:
        improvement = period.get('improvement', {}).get(improvement_name, {})
        display(Markdown(f'#### {improvement_name}'))
        display(pd.DataFrame([{'state': state, **improvement.get(state, {})} for state in ['-1', '0', '1']]).round(4))
    phase_improvement_rows = []
    for phase_name in ['full', 'development', 'validation', 'test']:
        phase_improvement = adjustment.get('periods', {}).get(phase_name, {}).get('improvement', {}).get('persistent_vs_base', {})
        for state in ['-1', '0', '1']:
            row = {'period': phase_name, 'state': state, **phase_improvement.get(state, {})}
            phase_improvement_rows.append(row)
    display(Markdown('### 持续持仓重标：全周期/分周期改善摘要（相对基线）'))
    display(pd.DataFrame(phase_improvement_rows).round(4))
else:
    display(Markdown('状态重构审计尚未形成完整双侧结果：请先运行两个侧别 Notebook；该审计不参与筛选。'))

display(Markdown('## 冻结信号路径图：颜色表示预测是否准确，表格给出对应 O2O 收益'))
signal_file = OUTPUT_DIR.parent / f'company_signal_{SIDE}.parquet'
if signal_file.is_file():
    signal = pd.read_parquet(signal_file)
    signal['formation_date'] = pd.to_datetime(signal['formation_date'])
    chart = panel[['formation_date', 'close', 'o2o_h1', 'o2o_h3', 'next_frozen_state']].copy()
    chart = chart.merge(signal[['formation_date', 'entry_signal', 'holding_signal']], on='formation_date', how='left')
    chart['entry_signal'] = chart['entry_signal'].fillna(0).astype(bool)
    chart['holding_signal'] = chart['holding_signal'].fillna(0).astype(bool)
    side_value = -1 if SIDE == 'down' else 1
    chart['directional_o2o_h1_bp'] = side_value * chart['o2o_h1'] * 10000.0
    chart['directional_o2o_h3_bp'] = side_value * chart['o2o_h3'] * 10000.0
    chart['correct'] = chart['directional_o2o_h1_bp'] > 0.0
    chart['phase'] = pd.cut(
        chart['formation_date'],
        bins=[pd.Timestamp('2017-12-31'), pd.Timestamp('2022-12-31'), pd.Timestamp('2024-12-31'), pd.Timestamp('2100-01-01')],
        labels=['development', 'validation', 'test'],
    )
    signal_rows = chart.loc[chart['entry_signal']].copy()
    summary_rows = []
    for phase_name in ['development', 'validation', 'test']:
        part = signal_rows.loc[signal_rows['phase'].eq(phase_name)]
        summary_rows.append({
            'period': phase_name,
            'signal_days': int(len(part)),
            'h1_correct_rate': float(part['correct'].mean()) if len(part) else float('nan'),
            'mean_directional_o2o_h1_bp': float(part['directional_o2o_h1_bp'].mean()) if len(part) else float('nan'),
            'mean_directional_o2o_h3_bp': float(part['directional_o2o_h3_bp'].mean()) if len(part) else float('nan'),
            'median_directional_o2o_h1_bp': float(part['directional_o2o_h1_bp'].median()) if len(part) else float('nan'),
            'p05_directional_o2o_h1_bp': float(part['directional_o2o_h1_bp'].quantile(0.05)) if len(part) else float('nan'),
            'p95_directional_o2o_h1_bp': float(part['directional_o2o_h1_bp'].quantile(0.95)) if len(part) else float('nan'),
        })
    display(Markdown('### 冻结 entry 信号分周期准确率与 O2O 收益（Test 只作冻结后观察）'))
    display(pd.DataFrame(summary_rows).set_index('period').round(2))
    display(Markdown('### 最近 30 个冻结 entry 信号：绿色=H1 方向正确，红色=方向错误'))
    table = signal_rows[['formation_date', 'phase', 'entry_signal', 'holding_signal', 'correct', 'directional_o2o_h1_bp', 'directional_o2o_h3_bp', 'next_frozen_state']].tail(30).copy()
    display(table.round({'directional_o2o_h1_bp': 2, 'directional_o2o_h3_bp': 2}))
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.plot(chart['formation_date'], chart['close'], color='#6b7280', linewidth=1.0, label='CSI500 close')
    correct = signal_rows['correct']
    ax.scatter(signal_rows.loc[correct, 'formation_date'], signal_rows.loc[correct, 'close'], color='#16a34a', marker='^' if SIDE == 'up' else 'v', s=42, label='entry H1 correct')
    ax.scatter(signal_rows.loc[~correct, 'formation_date'], signal_rows.loc[~correct, 'close'], color='#dc2626', marker='x', s=52, label='entry H1 wrong')
    annotate = signal_rows.tail(20)
    for _, point in annotate.iterrows():
        ax.annotate(f"{point['directional_o2o_h1_bp']:+.0f}bp", (point['formation_date'], point['close']), xytext=(0, 7), textcoords='offset points', fontsize=7, rotation=45, color='#111827')
    ax.set_title(f"{SIDE}: frozen entry signals on CSI500 close; green correct / red wrong; labels=directional O2O H1")
    ax.set_xlabel('formation date')
    ax.set_ylabel('close')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left')
    fig.tight_layout()
    display(fig)
    plt.close(fig)
else:
    display(Markdown('冻结信号文件尚未形成；请先运行本侧扫描。'))
